# 0. Imports

In [41]:
from datasets import load_dataset
from datasets import get_dataset_config_names
import pyarrow.parquet as pq
import pandas as pd
import polars as pl
import torch
from torch.utils.data import Dataset, DataLoader   
import torch.nn as nn
import torch.nn.functional as F 

get_dataset_config_names("HuggingFaceGECLM/REDDIT_comments")

['default']

# 1. Download and store Dataset

In [15]:
# import polars as pl
# from pathlib import Path

# N_ROWS = 1000000 # <- set this

# OUT = Path("data/processed/changemyview_permalink.parquet")
# OUT.parent.mkdir(parents=True, exist_ok=True)

# cols = [
#     "author", "body", "created_utc", "id", "link_id", "name",
#     "parent_id", "score", "controversiality", "total_awards_received"
# ]

# splits = {'AskHistorians': 'data/AskHistorians-*-of-*.parquet', 'DIY': 'data/DIY-*-of-*.parquet', 'Damnthatsinteresting': 'data/Damnthatsinteresting-*-of-*.parquet', 'Documentaries': 'data/Documentaries-*-of-*.parquet', 'EatCheapAndHealthy': 'data/EatCheapAndHealthy-*-of-*.parquet', 'Fantasy': 'data/Fantasy-*-of-*.parquet', 'Fitness': 'data/Fitness-*-of-*.parquet', 'Foodforthought': 'data/Foodforthought-00000-of-00001-9c1cfedaef23c26e.parquet', 'Games': 'data/Games-*-of-*.parquet', 'GetMotivated': 'data/GetMotivated-*-of-*.parquet', 'IAmA': 'data/IAmA-*-of-*.parquet', 'IWantToLearn': 'data/IWantToLearn-00000-of-00001-87aef57dd63767a8.parquet', 'LifeProTips': 'data/LifeProTips-*-of-*.parquet', 'Showerthoughts': 'data/Showerthoughts-*-of-*.parquet', 'SkincareAddiction': 'data/SkincareAddiction-*-of-*.parquet', 'UpliftingNews': 'data/UpliftingNews-*-of-*.parquet', 'WritingPrompts': 'data/WritingPrompts-*-of-*.parquet', 'YouShouldKnow': 'data/YouShouldKnow-*-of-*.parquet', 'askscience': 'data/askscience-*-of-*.parquet', 'bestof': 'data/bestof-*-of-*.parquet', 'boardgames': 'data/boardgames-*-of-*.parquet', 'bodyweightfitness': 'data/bodyweightfitness-*-of-*.parquet', 'books': 'data/books-*-of-*.parquet', 'buildapc': 'data/buildapc-*-of-*.parquet', 'changemyview': 'data/changemyview-*-of-*.parquet', 'explainlikeimfive': 'data/explainlikeimfive-*-of-*.parquet', 'femalefashionadvice': 'data/femalefashionadvice-*-of-*.parquet', 'gadgets': 'data/gadgets-*-of-*.parquet', 'gaming': 'data/gaming-*-of-*.parquet', 'gardening': 'data/gardening-*-of-*.parquet', 'history': 'data/history-*-of-*.parquet', 'ifyoulikeblank': 'data/ifyoulikeblank-*-of-*.parquet', 'lifehacks': 'data/lifehacks-*-of-*.parquet', 'malefashionadvice': 'data/malefashionadvice-*-of-*.parquet', 'mildlyinteresting': 'data/mildlyinteresting-*-of-*.parquet', 'personalfinance': 'data/personalfinance-*-of-*.parquet', 'philosophy': 'data/philosophy-*-of-*.parquet', 'podcasts': 'data/podcasts-00000-of-00001-9af2599ee776d606.parquet', 'programming': 'data/programming-*-of-*.parquet', 'relationship_advice': 'data/relationship_advice-*-of-*.parquet', 'science': 'data/science-*-of-*.parquet', 'scifi': 'data/scifi-*-of-*.parquet', 'socialskills': 'data/socialskills-*-of-*.parquet', 'space': 'data/space-*-of-*.parquet', 'sports': 'data/sports-*-of-*.parquet', 'suggestmeabook': 'data/suggestmeabook-*-of-*.parquet', 'technology': 'data/technology-*-of-*.parquet', 'tifu': 'data/tifu-*-of-*.parquet', 'todayilearned': 'data/todayilearned-*-of-*.parquet', 'travel': 'data/travel-*-of-*.parquet'}


# (
#     pl.scan_parquet("hf://datasets/HuggingFaceGECLM/REDDIT_comments/" + splits["changemyview"])
#     .select(cols)
#     .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
#     .limit(N_ROWS)                         # <- key line
#     .sink_parquet(OUT, compression="zstd")
# )

# print("Wrote:", OUT)


# 2. Load Datset from parquet file

In [16]:
df = (
    pl.scan_parquet("data/processed/changemyview_clean_head.parquet")
      .head(10_000)
      .collect()
      .to_pandas()
)

In [17]:
df.head()

,author,body,created_utc,id,link_id,name,parent_id,score,controversiality,total_awards_received
0,Thompson_S_Sweetback,1. Long term economic disincentives will not b...,1358441654,c7ymchk,t3_16ralh,t1_c7ymchk,t3_16ralh,14,0,NaN
1,Jaberkaty,Excellent points. I would only add that most g...,1358442499,c7ymmk3,t3_16ralh,t1_c7ymmk3,t1_c7ymchk,6,0,NaN
2,ancillarynipple,Where can you provide the data that people wit...,1358443385,c7ymwyx,t3_16ralh,t1_c7ymwyx,t1_c7ymchk,2,0,NaN
3,Thompson_S_Sweetback,Keynesian economics. I'm assuming that famili...,1358445013,c7ynh9o,t3_16ralh,t1_c7ynh9o,t1_c7ymwyx,4,0,NaN
4,ancillarynipple,In the spirit of this new subreddit I begrudgi...,1358447042,c7yo71x,t3_16ralh,t1_c7yo71x,t1_c7ynh9o,8,0,NaN


# 3. Prepare dataset

In [18]:
df = df.dropna(subset=["body"])

# convert string → number
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")

# now convert unix seconds → datetime
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# 2) Cutoff (80/20 temporal)
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test  = df[df["created_utc"] > cutoff].copy()

# 3) Hilfstabelle: comment_id -> author (nur innerhalb des Splits!)
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test  = df_test.set_index("id")["author"].to_dict()


In [19]:
def build_reply_pairs(df_split, id2author):
    # parent_id sieht oft so aus: "t1_xyz" oder "t3_abc"
    # wir wollen nur Replies auf Kommentare (t1_)
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # parent comment key extrahieren (ohne "t1_")
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # parent author mappen
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # valide Paare
    df_r = df_r.dropna(subset=["parent_author"])
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    pairs_pos = df_r[["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]].copy()
    pairs_pos = pairs_pos.rename(columns={"author": "u", "parent_author": "v", "id": "u_id", "parent_key": "v_id"})
    pairs_pos["y"] = 1
    return pairs_pos

pos_train = build_reply_pairs(df_train, id2author_train)
pos_test  = build_reply_pairs(df_test, id2author_test)

print("Pos train:", len(pos_train), "Pos test:", len(pos_test))

Pos train: 5559 Pos test: 1160


In [20]:
import numpy as np

def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    rng = np.random.default_rng(seed)

    # 1) Thread -> set(users)
    thread_users = df_split.groupby("link_id")["id"].apply(lambda s: set(s.dropna())).to_dict()

    # 2) Set aller reply-Kanten (und optional symmetrisch machen)
    #    damit negatives nicht aus echten reply-pairs stammen
    reply_edges = set(zip(pos_pairs["u_id"], pos_pairs["v_id"]))
    reply_edges_sym = reply_edges | set((v_id, u_id) for (u_id, v_id) in reply_edges)

    neg_rows = []
    pos_pairs_small = pos_pairs[["u_id", "v_id", "link_id"]].copy()

    for u_id, v_id, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Kandidaten: alle im Thread außer u; und nicht v falls du strenger sein willst
        cand = [x for x in users if x != u_id]
        if not cand:
            continue

        # filtere Kandidaten, die bereits reply-edge haben (in beide Richtungen)
        cand = [x for x in cand if (u_id, x) not in reply_edges_sym]
        if not cand:
            continue

        # sample k
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u_id, x, link_id, 0))

    neg = pd.DataFrame(neg_rows, columns=["u_id", "v_id", "link_id", "y"])
    return neg

neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test  = build_hard_negatives(df_test,  pos_test,  k_per_pos=2)

train_pairs = pd.concat([pos_train[["u_id","v_id","link_id","y"]], neg_train], ignore_index=True).sample(frac=1, random_state=42)
test_pairs  = pd.concat([pos_test[["u_id","v_id","link_id","y"]],   neg_test],  ignore_index=True).sample(frac=1, random_state=42)

print(train_pairs["y"].value_counts())
print(test_pairs["y"].value_counts())


y
0    11092
1     5559
Name: count, dtype: int64
y
0    2308
1    1160
Name: count, dtype: int64


In [21]:
train_pairs.to_parquet("data/processed/pairs_train.parquet", index=False)
test_pairs.to_parquet("data/processed/pairs_test.parquet", index=False)

In [22]:
# Anteil User ohne positives (sagt dir, wie „sparse“ es wirklich ist)
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print("Users with >=1 reply interaction (train):", len(pos_users), "/", len(all_users))

# Durchschnittliche positives pro u
print(pos_train.groupby("u").size().describe())


Users with >=1 reply interaction (train): 1058 / 1323
count    940.000000
mean       5.913830
std       22.311666
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max      610.000000
dtype: float64


In [23]:
# keine Selbstpaare
assert (train_pairs["u_id"] != train_pairs["v_id"]).all()

# negatives sind keine echten replies
real_edges = set(zip(pos_train["u_id"], pos_train["v_id"]))
assert not any(
    (u,v) in real_edges or (v,u) in real_edges
    for u,v in zip(
        neg_train["u_id"], neg_train["v_id"]
    )
)

# 4. Add text to user pairs

In [34]:
# Use df_train only to avoid leakage into training representations
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (optional) keep only active users to reduce noise
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

user_text_train = (
    df_train_text
    .sort_values("created_utc")             # ensures chronological order
    .groupby("id")["body"]
    .apply(lambda s: " ".join(s.tail(50)))  # last 50 comments per user (tune)
)

user_text_test = (
    df_test_text
    .sort_values("created_utc")
    .groupby("id")["body"]
    .apply(lambda s: " ".join(s.tail(50)))
)

# dict: user_id -> text
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

missing_train = train_pairs["u_id"].map(user_text_dict_train).isna().mean()
print("Missing user text for u_id:", missing_train)
missing_test = test_pairs["u_id"].map(user_text_dict_test).isna().mean()
print("Missing user text for u_id:", missing_test)


Missing user text for u_id: 0.0
Missing user text for u_id: 0.0


In [35]:
train_pairs = train_pairs.copy()
test_pairs  = test_pairs.copy()

def attach_text(pairs, user_text_dict):
    pairs["text_u"] = pairs["u_id"].map(user_text_dict)
    pairs["text_v"] = pairs["v_id"].map(user_text_dict)
    return pairs.dropna(subset=["text_u", "text_v"])

train_pairs_txt = attach_text(train_pairs, user_text_dict_train)
test_pairs_txt  = attach_text(test_pairs,  user_text_dict_test)

print(len(train_pairs), "->", len(train_pairs_txt))
print(len(test_pairs),  "->", len(test_pairs_txt))


16651 -> 16651
3468 -> 3468


# 5. Train CNN

### 5.1 Tokenize data

In [36]:
import re
from collections import Counter

TOKEN_RE = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build vocab from TRAIN texts only (no leakage)
counter = Counter()
for t in train_pairs_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in train_pairs_txt["text_v"].tolist():
    counter.update(tokenize(t))

# Special tokens
PAD = "<pad>"
UNK = "<unk>"

vocab = {PAD: 0, UNK: 1}
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print("Vocab size:", len(vocab))


Vocab size: 23432


### 5.2 Create Dataset

In [37]:
MAX_LEN = 256  # truncate long user docs

def encode(text: str):
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]

class PairDataset(Dataset):
    def __init__(self, df_pairs):
        self.u_texts = df_pairs["text_u"].tolist()
        self.v_texts = df_pairs["text_v"].tolist()
        self.y = df_pairs["y"].astype(float).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return encode(self.u_texts[idx]), encode(self.v_texts[idx]), self.y[idx]
    
def collate_fn(batch):
    u_seqs, v_seqs, ys = zip(*batch)
    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_lens = torch.tensor([len(s) for s in v_seqs], dtype=torch.long)

    max_u = max(u_lens).item()
    max_v = max(v_lens).item()

    u = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v = torch.full((len(batch), max_v), pad_id, dtype=torch.long)

    for i, s in enumerate(u_seqs):
        u[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    for i, s in enumerate(v_seqs):
        v[i, :len(s)] = torch.tensor(s, dtype=torch.long)

    y = torch.tensor(ys, dtype=torch.float32)
    return u, v, y

In [38]:
BATCH_SIZE = 128

train_ds = PairDataset(train_pairs_txt)
test_ds  = PairDataset(test_pairs_txt)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)


### 5.3 Create Siamese CNN

In [ ]:
class TextCNNEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, num_filters=128, kernel_sizes=(3,4,5), out_dim=128, pad_idx=0, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        # x: [B, T]
        emb = self.embedding(x)            # [B, T, E]
        emb = emb.transpose(1, 2)          # [B, E, T] for Conv1d

        conv_outs = []
        for conv in self.convs:
            h = F.relu(conv(emb))          # [B, F, T-k+1]
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)  # [B, F]
            conv_outs.append(h)

        h = torch.cat(conv_outs, dim=1)    # [B, F*len(K)]
        h = self.dropout(h)
        h = self.fc(h)                     # [B, out_dim]
        h = F.normalize(h, p=2, dim=1)     # unit vectors
        return h

class SiameseCNN(nn.Module):
    def __init__(self, encoder: nn.Module, scale=10.0):
        super().__init__()
        self.encoder = encoder
        self.scale = scale  # scales cosine to logits

    def forward(self, u, v):
        eu = self.encoder(u)
        ev = self.encoder(v)
        cos = (eu * ev).sum(dim=1)         # cosine, since normalized
        logits = self.scale * cos          # logit
        return logits

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = TextCNNEncoder(
    vocab_size=len(vocab),
    emb_dim=128,
    num_filters=128,
    kernel_sizes=(3,4,5),
    out_dim=128,
    pad_idx=pad_id,
    dropout=0.2,
)

model = SiameseCNN(encoder, scale=10.0).to(device)

### 5.4 Train the CNN and Evaluate

In [44]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)

@torch.no_grad()
def eval_auc(model, loader):
    model.eval()
    ys, ps = [], []
    for u, v, y in loader:
        u, v = u.to(device), v.to(device)
        logits = model(u, v)
        prob = torch.sigmoid(logits).cpu().numpy()
        ys.extend(y.numpy())
        ps.extend(prob)

    # AUC without sklearn (fast, stable)
    ys = np.array(ys)
    ps = np.array(ps)

    # rank-based AUC
    order = np.argsort(ps)
    ys_sorted = ys[order]
    n_pos = ys_sorted.sum()
    n_neg = len(ys_sorted) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = np.arange(1, len(ys_sorted) + 1)
    rank_sum_pos = ranks[ys_sorted == 1].sum()
    auc = (rank_sum_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg)
    return float(auc)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for u, v, y in loader:
        u, v, y = u.to(device), v.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(u, v)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)
    auc = eval_auc(model, test_loader)
    print(f"epoch={epoch}  loss={loss:.4f}  test_auc={auc:.4f}")


epoch=1  loss=1.1090  test_auc=0.5891
epoch=2  loss=0.8812  test_auc=0.6006
epoch=3  loss=0.8086  test_auc=0.6118
epoch=4  loss=0.7518  test_auc=0.6085
epoch=5  loss=0.6952  test_auc=0.6027
